# Conventional Machine Learning Models (Task 6)

## Purpose

Task 6 of Phase II requires the improved LCS-based system to be compared with
the original LCS and with at least three conventional, non-deep-learning
machine learning models. This notebook produces the conventional-model rows of
that comparison using **Logistic Regression**, **Random Forest** and
**Gaussian Naive Bayes**.

Every model is trained and evaluated through `src/evaluation.py`, the shared
evaluation module, so that all models in the project use:

* the same stratified 80/20 train-test split (`random_state=42`),
* the same stratified 10-fold cross-validation folds,
* the same set of evaluation metrics.

## Handling the class imbalance fairly

The target is imbalanced (86.07% no diabetes, 13.93% prediabetes/diabetes), so
each conventional model is run in two variants:

| Variant | Training data | Imbalance handling |
|---|---|---|
| A | Full training set (202,944 rows) | `class_weight='balanced'` |
| B | Majority class undersampled to 28,277 per class (56,554 rows) | Random undersampling, as in notebook 05 |
| C | Minority class oversampled to 174,667 per class (349,334 rows) | SMOTENC, as in notebook 05 |

Variant B matches the training data given to the improved eLCS system in
notebook 05, so it is the fair like-for-like comparison. Variant A shows what
the conventional models can achieve when given all available data, which the
LCS cannot practically use because of its runtime. Reporting both separates the
effect of the modelling method from the effect of the training-set size.

Variant C uses SMOTENC, which oversamples the minority class while keeping
categorical features valid: synthetic records take the most common category
among their neighbours instead of an interpolated value, so no impossible
values such as a smoking status of 0.6 are produced.

Undersampling and oversampling are applied through `BalancedUndersampler` and
`SMOTENCResampler`, which re-sample inside `fit()`. During cross-validation the balancing is therefore
re-applied to each training fold only and never touches the validation fold, so
no data leakage occurs.

## 1. Setup and shared split

In [ ]:
import sys
from pathlib import Path

sys.path.append("..")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from src.evaluation import (
    BalancedUndersampler,
    SMOTENCResampler,
    RANDOM_STATE,
    cross_validate_model,
    evaluate_on_test,
    friedman_test,
    get_cv_folds,
    load_cleaned_data,
    get_train_test_split,
    pairwise_wilcoxon,
    results_table,
)

sns.set_theme(style="whitegrid")

print("Modules imported. Shared random_state:", RANDOM_STATE)

In [ ]:
df = load_cleaned_data()

X_train, X_test, y_train, y_test = get_train_test_split(df)

print("Training set:", X_train.shape)
print("Test set:", X_test.shape)
print("\nTraining class counts:")
print(y_train.value_counts().sort_index())
print("\nTest class counts:")
print(y_test.value_counts().sort_index())

### Consistency check against notebook 05

The test set produced by the shared split must be identical to the test file
exported by notebook 05. If this check fails, the comparison in Task 6 would
not be evaluated on the same records and the results would not be comparable.

In [ ]:
elcs_test_path = Path("../data/processed/elcs/diabetes_elcs_test_unchanged.csv")

if elcs_test_path.exists():
    elcs_test = pd.read_csv(elcs_test_path)
    shared_test = X_test.copy()
    shared_test["Diabetes_binary"] = y_test

    same_shape = elcs_test.shape == shared_test.shape
    same_rows = np.array_equal(
        np.sort(elcs_test.to_numpy(), axis=0),
        np.sort(shared_test[elcs_test.columns].to_numpy(), axis=0),
    )

    print("Notebook 05 test file shape:", elcs_test.shape)
    print("Shared split test shape:    ", shared_test.shape)
    print("Shapes match:", same_shape)
    print("Same records:", same_rows)
else:
    print("Notebook 05 test file not found - merge Aman-branch first.")

## 2. Model selection and justification

Three conventional models are used, chosen so that they cover different
modelling assumptions rather than three variations of the same idea:

* **Logistic Regression** - a linear, additive baseline that is widely used in
  clinical risk scoring and is interpretable through its coefficients. It gives
  a reference point for how much predictive signal is available without any
  interaction effects.
* **Random Forest** - a non-linear ensemble that captures interactions between
  predictors automatically. It is the strongest expected competitor to the LCS
  and provides a demanding comparison.
* **Gaussian Naive Bayes** - a simple probabilistic model that assumes
  conditional independence between predictors. It is fast and provides a lower
  reference point that shows the value of modelling feature dependencies.

All three are non-deep-learning models, as required by the brief.

Logistic Regression and Gaussian Naive Bayes are sensitive to feature scale, so
they are wrapped in a `Pipeline` with `StandardScaler`. Placing the scaler in
the pipeline means it is fitted on training data only, including inside each
cross-validation fold. Random Forest is scale-invariant and needs no scaler.

In [ ]:
def make_models(balanced_class_weight):
    """
    Build the three conventional models.

    balanced_class_weight=True  -> variant A (cost-sensitive learning)
    balanced_class_weight=False -> variant B (undersampled training data)
    """
    class_weight = "balanced" if balanced_class_weight else None

    logistic_regression = Pipeline([
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(
            max_iter=1000,
            class_weight=class_weight,
            random_state=RANDOM_STATE,
        )),
    ])

    random_forest = RandomForestClassifier(
        n_estimators=300,
        min_samples_leaf=5,
        class_weight="balanced_subsample" if balanced_class_weight else None,
        n_jobs=-1,
        random_state=RANDOM_STATE,
    )

    naive_bayes = Pipeline([
        ("scaler", StandardScaler()),
        ("model", GaussianNB()),
    ])

    return {
        "Logistic Regression": logistic_regression,
        "Random Forest": random_forest,
        "Gaussian Naive Bayes": naive_bayes,
    }


models_variant_a = make_models(balanced_class_weight=True)

models_variant_b = {
    name: BalancedUndersampler(estimator, random_state=RANDOM_STATE)
    for name, estimator in make_models(balanced_class_weight=False).items()
}

# Every feature except BMI is categorical, matching notebook 05.
categorical_features = [c for c in X_train.columns if c != "BMI"]

models_variant_c = {
    name: SMOTENCResampler(
        estimator,
        categorical_features=categorical_features,
        random_state=RANDOM_STATE,
    )
    for name, estimator in make_models(balanced_class_weight=False).items()
}

print("Variant A (class weighting):", list(models_variant_a))
print("Variant B (undersampling): ", list(models_variant_b))
print("Variant C (SMOTENC):       ", list(models_variant_c))
print("\nCategorical features for SMOTENC:", len(categorical_features))

## 3. Held-out test set results

Each model is fitted on the training data and evaluated on the untouched 20%
test set, which keeps the original class distribution and therefore reflects
realistic screening conditions.

Predictions are stored because McNemar's test in notebook 07 compares models
record by record on this same test set.

In [ ]:
test_results = []
test_predictions = {}

for variant_label, models in [
    ("A: class weighting, full training set", models_variant_a),
    ("B: undersampled training set", models_variant_b),
    ("C: SMOTENC oversampled training set", models_variant_c),
]:
    for name, model in models.items():
        label = f"{name} ({variant_label[0]})"
        print(f"Fitting {label} ...")

        metrics, y_pred = evaluate_on_test(
            model, label, X_train, y_train, X_test, y_test
        )
        metrics["variant"] = variant_label

        test_results.append(metrics)
        test_predictions[label] = y_pred

        print(
            f"  balanced accuracy={metrics['balanced_accuracy']:.4f}  "
            f"recall={metrics['recall']:.4f}  "
            f"PR-AUC={metrics['pr_auc']:.4f}  "
            f"fit={metrics['fit_time_s']:.1f}s"
        )

In [ ]:
conventional_results = results_table(test_results)

display(
    conventional_results[[
        "accuracy", "balanced_accuracy", "precision",
        "recall", "f1", "roc_auc", "pr_auc",
    ]].round(4)
)

### Confusion matrices

The confusion matrices show how each model trades false positives against
missed cases. In a screening context a false negative (a missed
prediabetes/diabetes case) is generally more costly than a false positive,
which leads to a follow-up test rather than a missed diagnosis.

In [ ]:
n_models = len(test_results)
fig, axes = plt.subplots(3, 3, figsize=(16, 13))

for ax, metrics in zip(axes.ravel(), test_results):
    matrix = np.array([
        [metrics["tn"], metrics["fp"]],
        [metrics["fn"], metrics["tp"]],
    ])

    sns.heatmap(
        matrix,
        annot=True,
        fmt=",d",
        cmap="Blues",
        cbar=False,
        xticklabels=["No diabetes", "Pre/Diabetes"],
        yticklabels=["No diabetes", "Pre/Diabetes"],
        ax=ax,
    )
    ax.set_title(metrics["model"], fontsize=10)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")

for ax in axes.ravel()[n_models:]:
    ax.axis("off")

plt.tight_layout()
plt.show()

## 4. Cross-validation for the statistical tests

A single train-test split gives one score per model, which is not enough for a
statistical test. Stratified 10-fold cross-validation is therefore run inside
the training set, giving ten scores per model.

Ten folds are used rather than five because the Wilcoxon signed-rank test with
five paired observations cannot produce a two-sided p-value below 0.0625, so no
comparison could reach significance at the 5% level.

The folds come from `get_cv_folds()`, so the LCS models in the other notebooks
are evaluated on exactly the same partitions.

In [ ]:
folds = get_cv_folds(X_train, y_train)

cv_frames = []

variants = {
    "A": models_variant_a,
    "B": models_variant_b,
    "C": models_variant_c,
}

for variant, models in variants.items():
    for name, model in models.items():
        label = f"{name} ({variant})"
        print(f"Cross-validating {label} ...")

        cv_frames.append(
            cross_validate_model(
                model, label, X_train, y_train,
                folds=folds, verbose=False,
            )
        )

cv_results = pd.concat(cv_frames, ignore_index=True)

fold_summary = (
    cv_results
    .groupby("model")[["balanced_accuracy", "recall", "f1", "pr_auc"]]
    .agg(["mean", "std"])
    .round(4)
)

display(fold_summary)

In [ ]:
plt.figure(figsize=(11, 5))

sns.boxplot(
    data=cv_results,
    x="model",
    y="balanced_accuracy",
    hue="model",
    legend=False,
    palette="Blues",
)

plt.title("Balanced accuracy across 10 cross-validation folds")
plt.xlabel("")
plt.ylabel("Balanced accuracy")
plt.xticks(rotation=25, ha="right")
plt.tight_layout()
plt.show()

## 5. Statistical testing

Two tests are applied to the cross-validation scores:

* the **Friedman test**, a non-parametric test for whether any of the models
  differ across the folds. It is used because the same folds are applied to
  every model, so the scores are paired, and fold scores are not assumed to be
  normally distributed.
* **Wilcoxon signed-rank tests** on each pair of models, with a
  Holm-Bonferroni correction, to identify which specific pairs differ. The
  correction is needed because testing many pairs inflates the chance of a
  false positive.

In [ ]:
friedman = friedman_test(cv_results)

print("Friedman test on balanced accuracy across 10 folds")
print(f"  statistic = {friedman['statistic']:.4f}")
print(f"  p-value   = {friedman['p_value']:.6f}")
print("\nMean ranks (1 = best):")
display(friedman["mean_ranks"].round(3).to_frame("mean_rank"))

In [ ]:
wilcoxon = pairwise_wilcoxon(cv_results)

display(wilcoxon.round(4))

## 6. Export results for the Task 6 comparison

The metrics, fold scores and test-set predictions are written to `results/` so
that the full Task 6 comparison table, and McNemar's test against the LCS
models, can be assembled without re-running any model.

In [ ]:
# Compare the three imbalance strategies directly
variant_comparison = (
    conventional_results
    .reset_index()
    .assign(
        base_model=lambda d: d["model"].str.replace(r" \([ABC]\)$", "", regex=True),
        strategy=lambda d: d["model"].str.extract(r"\(([ABC])\)$"),
    )
    .pivot(index="base_model", columns="strategy", values="balanced_accuracy")
    .rename(columns={
        "A": "A: class weight",
        "B": "B: undersampled",
        "C": "C: SMOTENC",
    })
    .round(4)
)

print("Balanced accuracy by imbalance strategy\n")
print(variant_comparison)

print("\nTraining rows used: A = 202,944  B = 56,554  C = 349,334")

In [ ]:
results_dir = Path("../results")
predictions_dir = results_dir / "predictions"
predictions_dir.mkdir(parents=True, exist_ok=True)

conventional_results.to_csv(results_dir / "conventional_test_metrics.csv")
variant_comparison.to_csv(results_dir / "imbalance_strategy_comparison.csv")
cv_results.to_csv(results_dir / "conventional_cv_scores.csv", index=False)
wilcoxon.to_csv(results_dir / "conventional_wilcoxon.csv", index=False)

for label, y_pred in test_predictions.items():
    filename = label.replace(" ", "_").replace("(", "").replace(")", "")
    np.save(predictions_dir / f"{filename}.npy", y_pred)

np.save(predictions_dir / "y_test.npy", np.asarray(y_test).astype(int))

print("Saved to ../results/:")
for path in sorted(results_dir.rglob("*")):
    if path.is_file():
        print(" ", path.relative_to(results_dir))

## 7. Findings

*Complete this section after running the notebook.*

Points to address:

* Which model achieved the best balanced accuracy, recall and PR-AUC, and
  whether the ranking changes depending on the metric used.
* How the three imbalance strategies compare. If variant A and variant B are
  close, the LCS is not disadvantaged by training on the smaller balanced set,
  which strengthens the fairness of the Task 6 comparison. If variant A is
  clearly better, the 146,390 discarded majority records carried useful
  information and this must be acknowledged as a limitation.
* Whether SMOTENC (variant C) outperforms random undersampling (variant B).
  Undersampling discards real records while SMOTENC synthesises new ones, so
  this comparison shows whether the synthetic minority records carry useful
  information or mainly add noise. This directly informs which strategy the
  improved LCS system should use.
* Whether the Friedman test indicates any difference between the models, and
  which pairs the corrected Wilcoxon tests identify as differing.
* Whether statistically significant differences are also practically
  meaningful. With a test set of 50,736 records, very small differences can
  reach significance without mattering for screening decisions.
* How the conventional models compare with the original eLCS baseline from
  notebook 04, which reached 50.20% balanced accuracy and 0.45% recall.